In [1]:
import sys
sys.path.append('../')

In [2]:
from ice_fx.infrastructure.instrument_collection import instrumentCollection as ic
from ice_fx.oanda_api.oanda_api import OandaApi
from ice_fx.infrastructure.collect_data import run_collection
from findicator import (atr, bollinger_bands, macd, impulse_system, ema_channel,
                        elder_ray, stochastic_oscillator, force_index, rsi)
# from findicator import stochastic_oscillator
from chronofusion.utils.errors import InvalidArgumentError, NonOverlappingData, NotAvailableData

api = OandaApi()
ic.create_file(api.get_account_instruments(), "./test_env_data")
ic.load_instruments("./test_env_data")
# ic.print_instruments()
from collections import deque, namedtuple
import torch
import torch.nn as nn
import time




# Environment

In [3]:
import pandas as pd
m5_data = pd.read_pickle('./test_env_data/EUR_USD_M5.pkl')
m30_data = pd.read_pickle('./test_env_data/EUR_USD_M30.pkl')

In [4]:
m30_data_filtered = m30_data[['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c']].copy()
m5_data_filtered = m5_data[['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c']].copy()

## Window Generator

In [5]:
class TimeFrameWindowGen:
    def __init__(self, 
                 higher_tf_data:pd.DataFrame, 
                 lower_tf_data:pd.DataFrame, 
                 higher_window_size:int = 50, 
                 lower_window_size:int = 50,
                 htf_start_index=0):
        
        # self.timestep = 0
        self.htf_data = higher_tf_data
        self.ltf_data = lower_tf_data
        self.h_window_size = higher_window_size
        self.l_window_size = lower_window_size
        self.alignment(htf_start_index=htf_start_index)

    def alignment(self, htf_start_index: int):
        self.higher_last_candle_index = self.h_window_size - 1 + htf_start_index
        if self.higher_last_candle_index >= self.htf_data.shape[0]:
            raise NotAvailableData(index=self.higher_last_candle_index, data_frame_scale='higher')
        time_to_align = self.htf_data['time'].iloc[self.higher_last_candle_index]
        # print(self.htf_data.iloc[self.higher_last_candle_index])
        aligned_index = self.ltf_data.index[self.ltf_data["time"] == time_to_align]
        if aligned_index.empty:
            raise NonOverlappingData()
        else:
            self.lower_last_candle_index = aligned_index[0]

    def generate(self):
        
        start = self.higher_last_candle_index - self.h_window_size + 1
        end = self.higher_last_candle_index + 1
        if end >= self.htf_data.shape[0]:
            raise NotAvailableData(index=end, data_frame_scale='higher')
        if start >= self.htf_data.shape[0]:
            raise NotAvailableData(index=start, data_frame_scale='higher')
        
        higher_fraction = self.htf_data[start:end]


        start = self.lower_last_candle_index - self.l_window_size + 1 
        end = self.lower_last_candle_index + 1
        if end >= self.ltf_data.shape[0]:
           raise NotAvailableData(index=end, data_frame_scale='lower')
        if start >= self.ltf_data.shape[0]:
           raise NotAvailableData(index=start, data_frame_scale='lower')
        
        lower_fraction = self.ltf_data[start:end]
            
        # self.timestep += 1
        self.lower_last_candle_index += 1

        higher_time_delta = self.htf_data['time'].iloc[self.higher_last_candle_index] - \
                            self.htf_data['time'].iloc[self.higher_last_candle_index-1]
        
        if self.htf_data['time'].iloc[self.higher_last_candle_index] + higher_time_delta <=\
           self.ltf_data['time'].iloc[self.lower_last_candle_index]:
            
            self.higher_last_candle_index += 1
        

        return higher_fraction, lower_fraction
    
    def get_current_price(self):
        curr_price_index  = self.lower_last_candle_index - 1 
        return self.ltf_data['mid_c'].iloc[curr_price_index]
        

In [6]:
window_generator = TimeFrameWindowGen(m30_data_filtered, m5_data_filtered,
                                      higher_window_size=5, lower_window_size=5)

for i in range(20):

    if i == 10:
        print('\n\n\n')
        window_generator.alignment(htf_start_index=5000)
        
    h, l = window_generator.generate()
    
    curr_price = window_generator.get_current_price()
    print(h, '\n')
    print(l, '\n')
    print(curr_price, '\n *************************')
    

                       time  volume    mid_o    mid_h    mid_l    mid_c
0 2016-01-07 00:00:00+00:00     335  1.07764  1.07832  1.07759  1.07790
1 2016-01-07 00:30:00+00:00     207  1.07786  1.07810  1.07744  1.07778
2 2016-01-07 01:00:00+00:00    1323  1.07776  1.07990  1.07748  1.07966
3 2016-01-07 01:30:00+00:00    1844  1.07968  1.08100  1.07931  1.08029
4 2016-01-07 02:00:00+00:00     854  1.08026  1.08086  1.07996  1.08018 

                        time  volume    mid_o    mid_h    mid_l    mid_c
20 2016-01-07 01:40:00+00:00     334  1.08027  1.08099  1.08027  1.08079
21 2016-01-07 01:45:00+00:00     387  1.08076  1.08097  1.07937  1.07992
22 2016-01-07 01:50:00+00:00     369  1.07994  1.08100  1.07992  1.08100
23 2016-01-07 01:55:00+00:00     267  1.08098  1.08098  1.08029  1.08029
24 2016-01-07 02:00:00+00:00     210  1.08026  1.08068  1.08026  1.08027 

1.08027 
 *************************
                       time  volume    mid_o    mid_h    mid_l    mid_c
0 2016-01-07 00:00

## Env

In [7]:
import numpy as np
import pandas as pd
from collections import deque
import gym
from gym import spaces
import talib
from typing import Tuple, Dict, List
import warnings
warnings.filterwarnings('ignore')

class MultiTimeframeForexEnv(gym.Env):
    """
    Multi-Timeframe Forex Trading Environment for DQN
    Actions: 0=close, 1=buy, 2=sell, 3=hold
    Termination: 2% loss or 6% profit on trade
    Features: Prevents future time leakage, supports walk-forward training
    """
    
    def __init__(self, 
                 lower_tf_data: pd.DataFrame,
                 higher_tf_data: pd.DataFrame,
                #  timeframe_ratio: int = 5,  # e.g., 5min/1min = 5
                 initial_balance: float = 10000, 
                 lot_size: float = 0.1,
                 stop_loss_pct: float = 0.02, 
                 take_profit_pct: float = 0.06,
                 lookback_window: int = 20, 
                 transaction_cost: float = 0.0001,
                 walk_forward_window: int = 2000,  # Training window size
                 #validation_window: int = 500
                 ):    # Validation window size
        
        super(MultiTimeframeForexEnv, self).__init__()
        
        # Data preparation
        self.lower_tf_data = lower_tf_data.reset_index(drop=True).copy()
        self.higher_tf_data = higher_tf_data.reset_index(drop=True).copy()
        self._add_indicators()

        self.window_generator = TimeFrameWindowGen(self.higher_tf_data, 
                                                   self.lower_tf_data,
                                                   higher_window_size=lookback_window, 
                                                   lower_window_size=lookback_window,
                                                   htf_start_index=0)
        # self.timeframe_ratio = timeframe_ratio
        
        # Trading parameters
        self.initial_balance = initial_balance
        self.lot_size = lot_size
        self.stop_loss_pct = stop_loss_pct
        self.take_profit_pct = take_profit_pct
        self.lookback_window = lookback_window
        self.transaction_cost = transaction_cost
        
        # Walk-forward parameters
        self.walk_forward_window = walk_forward_window
        # self.validation_window = validation_window
        self.training_start = 0
        self.training_end = walk_forward_window
        # self.is_training = True
        
        
        # Position tracking
        self.current_position = 0.0
        self.entry_price = None
        self.current_step = 0
        self.current_balance = initial_balance
        self.trade_history = []
        self.steps_in_trade = 0
        
        # Action space: 0=close, 1=buy, 2=sell, 3=hold
        self.action_space = spaces.Discrete(4)
        
        # State space (combo function output)
        # self.observation_space = spaces.Box(
        #     low=-np.inf, high=np.inf, 
        #     shape=(10,), dtype=np.float32
        # )
        
        # Episode tracking
        self.max_steps = len(self.lower_tf_data) - self.lookback_window - 1
        
        # Indicators cache for preventing future leakage
        # self.lower_indicators_cache = {}
        # self.higher_indicators_cache = {}
        
    
    def _add_indicators(self) -> pd.DataFrame:  # OK
        
        
        close_price = self.lower_tf_data['mid_c'].copy()
        high_price = self.lower_tf_data['mid_h'].copy()
        low_price = self.lower_tf_data['mid_l'].copy()
        volume = self.lower_tf_data['volume']

        self.lower_tf_data['force_index'] = force_index(close=close_price, volume=volume)
        # print(elder_ray(high=high_price, low=low_price, close=close_price).shape)
        self.lower_tf_data['elder_ray_bull'], self.lower_tf_data['elder_ray_bear'] = elder_ray(high=high_price, low=low_price, close=close_price)
        self.lower_tf_data['stochastic_k'], self.lower_tf_data['stochastic_d'] = stochastic_oscillator(high=high_price, low=low_price, close=close_price)
        self.lower_tf_data['ris'] = rsi(close=close_price)
        
        
        close_price = self.higher_tf_data['mid_c'].copy()
        
        self.higher_tf_data['ema_26'] = ema_channel(close_price, ema_period=26, use_atr=False)
        self.higher_tf_data['ema_13'] = ema_channel(close_price, ema_period=13, use_atr=False)
        _, _, self.higher_tf_data['macd_hist'] =  macd(close=close_price)
        self.higher_tf_data['impulse_system'] = impulse_system(close=close_price)

        
        self.lower_tf_data.dropna(inplace=True)
        self.higher_tf_data.dropna(inplace=True)
    

    def set_walk_forward_window(self, start: int, end: int): # OK is_training: bool = True
        """Set the walk-forward training/validation window"""
        # self.training_start = start
        # self.training_end = end
        # self.is_training = is_training
        
        self.window_generator.alignment(htf_start_index=start)
        # # Clear indicators cache when switching windows
        # self.lower_indicators_cache.clear()
        # self.higher_indicators_cache.clear()
        
        print(f"Set training window: {start}-{end}")
    
    def reset(self): # OK 
        """Reset environment to initial state"""
        self.current_balance = self.initial_balance
        self.current_position = 0.0
        self.entry_price = None
        self.current_step = 0
        # Start from training window beginning + lookback
        # if self.is_training:
        #     self.current_step = max(self.training_start + self.lookback_window, self.lookback_window)
        # else:
        #     self.current_step = max(self.training_end + self.lookback_window, self.lookback_window)
        
        self.trade_history = []
        self.steps_in_trade = 0
        
        return self._get_state()
    
    def _get_state(self): # OK
        """Get current state using combo function"""
        
        higher_candles, lower_candles = self.window_generator.generate()
        # higher_candles.drop(columns=["time"], inplace=True)
        # lower_candles.drop(columns=["time"], inplace=True)

        #state = self.combo(higher_candles, lower_candles)
        
        # data = (higher_candles, lower_candles)
        # Add position information
        position_info = np.array([
            self.current_position / self.lot_size,  # -1, 0, or +1
            self._calculate_unrealized_pnl() / self.initial_balance,  # normalized unrealized P&L
        ], dtype=np.float32)
        
        # Replace last 2 features with position info
        # state[-2:] = position_info

        state = {'ltf_data':lower_candles,
                 'htf_data': higher_candles,
                 'position_ifno':position_info}
        
        return state
    
    def _calculate_unrealized_pnl(self): # OK
        """Calculate unrealized profit/loss"""
        if self.current_position == 0.0 or self.entry_price is None:
            return 0.0
        
        # current_price = self.lower_tf_data.iloc[self.current_step]['mid_c']
        current_price = self.window_generator.get_current_price()
        
        if self.current_position > 0:  # Long position
            pnl = (current_price - self.entry_price) * abs(self.current_position) * 100000
        else:  # Short position
            pnl = (self.entry_price - current_price) * abs(self.current_position) * 100000
            
        return pnl
    
    def _get_valid_actions(self): # OK
        """Get valid actions based on current position"""
        if self.current_position == 0.0:  # Flat
            return [0, 1, 2]  # close(no-op), buy, sell
        else:  # In position
            return [0, 3]  # close, hold
    
    def step(self, action):
        """Execute one step in the environment"""
        # current_price = self.lower_tf_data.iloc[self.current_step]['close']
        current_price = self.window_generator.get_current_price()
        # print('prince before action:', current_price)
        # Execute action
        reward = self._execute_action(action, current_price)
        
        # Check termination conditions
        done = self._check_termination(current_price)
        reward += done
        done = True if done in [100.0, -100.0] else False

        pos_type = None
        # Move to next step
        self.current_step += 1
        if self.current_position != 0:
            if self.current_position>0:
                pos_type = 'Long'
            else:
                pos_type = 'Short'
            self.steps_in_trade += 1

        else:
            pos_type = 'No Position'
        
        # Check if we've reached end of current window
        # if self.is_training:
        #     max_step = self.training_end - 1
        # else:
        #     max_step = min(self.training_end + self.validation_window - 1, self.max_steps)
        
        if self.current_step >= self.walk_forward_window: # max_step:
            done = True
            # Force close any open position
            if self.current_position != 0:
                reward += self._close_position(current_price)
        
        next_state = self._get_state() if not done else np.zeros(10)
        # print('prince after action:', self.window_generator.get_current_price())
        # print("*"*20, self.is_training)
        
        info = {
            'current_balance': self.current_balance,
            'position_type': pos_type,
            'current_position_size': self.current_position,
            'unrealized_pnl': self._calculate_unrealized_pnl(),
            'steps_in_trade': self.steps_in_trade,
            # 'is_training': self.is_training,
            # 'window_progress': (self.current_step - (self.training_start if self.is_training else self.training_end)) / 
            #                  (self.training_end - self.training_start if self.is_training else self.validation_window)
        }
        
        return next_state, reward, done, info
    
    def _execute_action(self, action, current_price):  # OK
        """Execute the given action and return reward"""
        valid_actions = self._get_valid_actions()
        
        if action not in valid_actions:
            return -0.01  # Invalid action penalty
        
        reward = 0.0
        
        if action == 0:  # Close position
            if self.current_position != 0:
                reward = self._close_position(current_price)
            
        elif action == 1:  # Buy
            if self.current_position == 0:
                reward = self._open_position(current_price, self.lot_size)
            
        elif action == 2:  # Sell
            if self.current_position == 0:
                reward = self._open_position(current_price, -self.lot_size)
                
        elif action == 3:  # Hold
            if self.current_position != 0:
                # Small time penalty + unrealized P&L signal
                unrealized_pnl = self._calculate_unrealized_pnl()
                reward = -0.0001 + (unrealized_pnl / self.initial_balance) * 0.01
            
        return reward
    
    def _open_position(self, price, position_size):  # OK 
        """Open a new position"""
        self.current_position = position_size
        self.entry_price = price
        self.steps_in_trade = 0
        
        # Transaction cost
        transaction_cost = abs(position_size) * 100000 * self.transaction_cost
        # print('Transaction cost', transaction_cost)
        self.current_balance -= transaction_cost
        
        return -0.001  # Small penalty for opening position
    
    def _close_position(self, price): # OK
        """Close current position and return realized P&L"""
        if self.current_position == 0:
            return 0.0
        # print('Entry price: ', self.entry_price)
        # print('Quit price: ', price)
        # Calculate realized P&L
        if self.current_position > 0:  # Long position
            pnl = (price - self.entry_price) * abs(self.current_position) * 100000
        else:  # Short position
            pnl = (self.entry_price - price) * abs(self.current_position) * 100000
        
        # Transaction cost
        transaction_cost = abs(self.current_position) * 100000 * self.transaction_cost
        net_pnl = pnl - transaction_cost
        
        # Update balance
        self.current_balance += net_pnl
        
        # Record trade
        self.trade_history.append({
            'entry_price': self.entry_price,
            'exit_price': price,
            'position':'Long' if self.current_position>0 else 'Short',
            'position_size': self.current_position,
            'pnl': net_pnl,
            'steps_in_trade': self.steps_in_trade,
            # 'entry_step': self.current_step - self.steps_in_trade,
            'exit_step': self.current_step
        })
        
        # Reset position
        self.current_position = 0.0
        self.entry_price = None
        self.steps_in_trade = 0
        
        # Return normalized reward
        return net_pnl / self.initial_balance
    
    def _check_termination(self, current_price): # Ok
        """Check if episode should terminate"""
        if self.current_position == 0:
            return 0 # False
        
        unrealized_pnl = self._calculate_unrealized_pnl()
        # trade_value = abs(self.current_position) * 100000 * self.entry_price
        
        # Stop loss: 2% loss
        if unrealized_pnl <= -self.initial_balance * self.stop_loss_pct: # -trade_value * self.stop_loss_pct:
            self._close_position(current_price)
            return -100.0
        
        # Take profit: 6% profit
        if unrealized_pnl >= self.initial_balance * self.take_profit_pct: # trade_value * self.take_profit_pct:
            self._close_position(current_price)
            return 100.0
        
        return 0 # False

    def action_to_name(self,act_num:int):
        act_dict = {0:'Close/No Op',
                    1:"Buy",
                    2:"Sell",
                    3:"Hold"}
        if act_num not in act_dict.keys():
            print(f"{act_num} is not a valid action!")
        else:
            return act_dict[act_num]
        
    def get_current_price(self):
        return self.window_generator.get_current_price()
    
    def get_trade_statistics(self):
        """Get trading statistics"""
        if not self.trade_history:
            return {}
        
        trades_df = pd.DataFrame(self.trade_history)
        
        stats = {
            'total_trades': len(trades_df),
            'winning_trades': len(trades_df[trades_df['pnl'] > 0]),
            'losing_trades': len(trades_df[trades_df['pnl'] < 0]),
            'total_pnl': trades_df['pnl'].sum(),
            'average_pnl': trades_df['pnl'].mean(),
            'max_profit': trades_df['pnl'].max(),
            'max_loss': trades_df['pnl'].min(),
            'win_rate': len(trades_df[trades_df['pnl'] > 0]) / len(trades_df) * 100,
            'average_trade_duration': trades_df['steps_in_trade'].mean(),
            'sharpe_ratio': trades_df['pnl'].mean() / (trades_df['pnl'].std() + 1e-8) if len(trades_df) > 1 else 0.0,
            'final_balance': self.current_balance
        }
        
        return stats

env = MultiTimeframeForexEnv(
        lower_tf_data=m5_data_filtered,
        higher_tf_data=m30_data_filtered,
        # timeframe_ratio=5,
        walk_forward_window=2000,
        # validation_window=500
    )

## Walk Forward Generator

In [8]:
def create_walk_forward_trainer(
                               total_steps: int,
                               window_size: int = 2000,
                               overlap: int = 200):
    """
    Create walk-forward training windows
    
    Example:
    - Window 1: Train 0-2000, 
    - Window 2: Train 1800-3800, 
    - Window 3: Train 3600-5600
    """
    windows = []
    start = 0
    
    while start + window_size  < total_steps:
        train_start = start
        train_end = start + window_size
        
        windows.append({
            'train_start': train_start,
            'train_end': train_end,
        })
        
        start += window_size - overlap
    
    return windows


## Testing Environment

In [9]:
lower_tf_data = m5_data_filtered
higher_tf_data = m30_data_filtered

env = MultiTimeframeForexEnv(
    lower_tf_data=lower_tf_data,
    higher_tf_data=higher_tf_data,
    walk_forward_window=2000,
)


windows = create_walk_forward_trainer(
    total_steps=8000,
    window_size=2000,
    overlap=200
)

# print(f"\n=== Walk-Forward Training Windows ===")
# for i, window in enumerate(windows):
#     print(f"Window {i+1}:")
#     print(f"  Train: {window['train_start']}-{window['train_end']}")

# Test first training window
# print(f"\n=== Testing Window 1 ===")

window = windows[0]
env.set_walk_forward_window(window['train_start'], window['train_end'])

state = env.reset()
# print('start price:', env.get_current_price())
# print(f"State shape:\n{state['data'][1]['mid_c'].tail()}\n")

# Run training simulation
total_reward = 0
steps = 0
trades = 0

for step in range(50):  # Test first 50 steps
    # Get valid actions and choose one
    valid_actions = env._get_valid_actions()
    # print('valid actions:', valid_actions)
    action = np.random.choice(valid_actions)
    # print('chosen action', env.action_to_name(action))
    next_state, reward, done, info = env.step(action)
    # print('\n\n')
    # print('current price:', env.get_current_price(), '\n')
    total_reward += reward
    steps += 1
    
    if len(env.trade_history) > trades:
        trades = len(env.trade_history)
        print(f"Step {steps}: Trade completed! Position={env.trade_history[-1]['position']}, Reward={reward:.6f}")
        print(f"Entry Price:{env.trade_history[-1]['entry_price']}, Entry Price:{env.trade_history[-1]['exit_price']}, Balance: ${info['current_balance']:.2f}\n")
        # print(f"  Position: {info['current_position_size']}")
        # print(f"  Window Progress: {info['window_progress']:.1%}")
    
    if done:
        print(f"Episode terminated at step {steps}")
        break

Set training window: 0-2000
Step 5: Trade completed! Position=Long, Reward=0.000170
Entry Price:1.08474, Entry Price:1.08501, Balance: $10000.70

Step 9: Trade completed! Position=Short, Reward=-0.001220
Entry Price:1.08504, Entry Price:1.08616, Balance: $9987.50

Step 12: Trade completed! Position=Short, Reward=0.000410
Entry Price:1.08689, Entry Price:1.08638, Balance: $9990.60

Step 18: Trade completed! Position=Long, Reward=-0.001500
Entry Price:1.08706, Entry Price:1.08566, Balance: $9974.60

Step 20: Trade completed! Position=Long, Reward=0.000390
Entry Price:1.08526, Entry Price:1.08575, Balance: $9977.50

Step 24: Trade completed! Position=Short, Reward=0.000510
Entry Price:1.08525, Entry Price:1.08464, Balance: $9981.60

Step 26: Trade completed! Position=Short, Reward=-0.000110
Entry Price:1.08549, Entry Price:1.0855, Balance: $9979.50

Step 28: Trade completed! Position=Long, Reward=0.000840
Entry Price:1.08584, Entry Price:1.08678, Balance: $9986.90

Step 33: Trade complete

# Agent

In [10]:
# src/chronofusion/models/transformer.py


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x


class SingleScaleEncoder(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True  # ✅ Now everything is (batch, seq_len, embed_dim)
        )
        self.transformer_encoder = nn.TransformerEncoder(
            self.encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):
        """
        x: (batch_size, seq_len, input_dim)
        """
        x = self.input_proj(x)
        x = self.pos_encoder(x)

        seq_len = x.size(1)
        device = x.device

        # Causal mask: (seq_len, seq_len)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=device) * float('-inf'),
            diagonal=1
        )

        out = self.transformer_encoder(x, mask=causal_mask)  # Now batch_first=True

        return out

class MultiScaleTransformer(nn.Module):
    def __init__(
        self,
        input_dim_low,
        input_dim_high,
        d_model,
        nhead,
        num_layers,
        output_dim=100
    ):
        super().__init__()
        self.low_encoder = SingleScaleEncoder(input_dim_low, d_model, nhead, num_layers)
        self.high_encoder = SingleScaleEncoder(input_dim_high, d_model, nhead, num_layers)

        # Fusion: concat both representations per timestep + projection to output_dim
        self.fusion_proj = nn.Linear(d_model * 2, output_dim)

    def forward(self, x_low, x_high):
        """
        x_low: (batch_size, seq_len, input_dim_low)  # e.g., 30min timeframe window
        x_high: (batch_size, seq_len, input_dim_high) # e.g., 5min timeframe window
        """
        enc_low = self.low_encoder(x_low)    # (batch_size, seq_len, d_model)
        enc_high = self.high_encoder(x_high) # (batch_size, seq_len, d_model)

        combined = torch.cat([enc_low, enc_high], dim=-1)  # (batch_size, seq_len, d_model*2)
        out = self.fusion_proj(combined)                   # (batch_size, seq_len, output_dim=100)
        out = out.mean(dim=1)
        return out


In [11]:
torch.cuda.is_available()

False

In [12]:
lower_tf_data = m5_data_filtered
higher_tf_data = m30_data_filtered

env = MultiTimeframeForexEnv(
    lower_tf_data=lower_tf_data,
    higher_tf_data=higher_tf_data,
    walk_forward_window=2000,
)


windows = create_walk_forward_trainer(
    total_steps=8000,
    window_size=2000,
    overlap=200
)


window = windows[0]
env.set_walk_forward_window(window['train_start'], window['train_end'])

state = env.reset()

# total_reward = 0
# steps = 0
# trades = 0

# for step in range(50): 
#     valid_actions = env._get_valid_actions()
#     action = np.random.choice(valid_actions)
#     next_state, reward, done, info = env.step(action)
#     total_reward += reward
#     steps += 1

Set training window: 0-2000


In [13]:
ltf_window = state['ltf_data']
htf_window = state['htf_data']

agent = MultiScaleTransformer(input_dim_high=htf_window.shape[1]-1, input_dim_low=ltf_window.shape[1]-1,
                              d_model=256, nhead=4, num_layers=5, output_dim=100)


ltf_numeric = ltf_window.select_dtypes(include=["number"])
htf_numeric = htf_window.select_dtypes(include=["number"])

x_low = torch.tensor(ltf_numeric.values, dtype=torch.float32)
x_high = torch.tensor(htf_numeric.values, dtype=torch.float32)

out = agent(x_low.unsqueeze(0), x_high.unsqueeze(0))
out.shape

torch.Size([1, 100])

## FinalModel

In [14]:
import torch
import torch.nn as nn

class QModel(nn.Module):
    def __init__(self, 
                 input_dim_low, input_dim_high, 
                 d_model=256, nhead=4, num_layers=5,
                 fusion_dim=100,
                 pos_info_dim=10):  # <- length of extra array
        super().__init__()

        # Multi-scale transformer backbone
        self.backbone = MultiScaleTransformer(
            input_dim_low=input_dim_low,
            input_dim_high=input_dim_high,
            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            output_dim=fusion_dim
        )

        # Regression head: (100 + extra_dim) → ... → 4
        self.reg_head = nn.Sequential(
            nn.Linear(fusion_dim + pos_info_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 4)   # 4 outputs for regression
        )

    def forward(self, x_low, x_high, extra_array):
        """
        x_low: (batch, seq_len, input_dim_low)
        x_high: (batch, seq_len, input_dim_high)
        extra_array: (batch, extra_dim)
        """
        fused = self.backbone(x_low, x_high)  # (batch, 100)

        # concat with extra input
        combined = torch.cat([fused, extra_array], dim=-1)  # (batch, 100+extra_dim)

        # regression output
        out = self.reg_head(combined)  # (batch, 4)

        return out


In [15]:
# suppose extra feature vector has length 8


# batch_size = 2
# x_low = torch.randn(batch_size, 20, 10)   # ltf_window
# x_high = torch.randn(batch_size, 20, 11)  # htf_window
# extra = torch.randn(batch_size, 8)        # extra array

ltf_window = state['ltf_data']
htf_window = state['htf_data']

ltf_numeric = ltf_window.select_dtypes(include=["number"])
htf_numeric = htf_window.select_dtypes(include=["number"])

x_low = torch.tensor(ltf_numeric.values, dtype=torch.float32)
x_high = torch.tensor(htf_numeric.values, dtype=torch.float32)
extra = torch.tensor(state['position_ifno'], dtype=torch.float32)


model = QModel(
    input_dim_low=x_low.shape[1], 
    input_dim_high=x_high.shape[1],
    pos_info_dim=extra.shape[0]
)

out = model(x_low.unsqueeze(0), x_high.unsqueeze(0), extra.unsqueeze(0))
print(out.shape)  # torch.Size([2, 4])


torch.Size([1, 4])


In [16]:
out

tensor([[-0.0887,  0.1853,  0.1148, -0.0356]], grad_fn=<AddmmBackward0>)

# DQN

In [17]:
MEMORY_SIZE = 100_000     # size of memory buffer
GAMMA = 0.995             # discount factor
ALPHA = 1e-3              # learning rate  
NUM_STEPS_FOR_UPDATE = 4  # perform a learning update every C time steps
LOOKBACK_WINDOW_SIZE = 26  

In [18]:
# state_size = env.observation_space.shape
num_actions = env.action_space.n
# print(num_actions)

In [19]:
# m30_data_filtered[-10:-1]

In [20]:
# m5_data_filtered[-10:-1]

In [21]:
def state_to_tensor(state):
    ltf_window = state['ltf_data']
    htf_window = state['htf_data']

    ltf_numeric = ltf_window.select_dtypes(include=["number"])
    htf_numeric = htf_window.select_dtypes(include=["number"])

    x_low = torch.tensor(ltf_numeric.values, dtype=torch.float32)
    x_high = torch.tensor(htf_numeric.values, dtype=torch.float32)
    pos_info = torch.tensor(state['position_ifno'], dtype=torch.float32)

    return x_low, x_high, pos_info

In [22]:
lower_tf_data = m5_data_filtered
higher_tf_data = m30_data_filtered


env = MultiTimeframeForexEnv(
    lower_tf_data=lower_tf_data,
    higher_tf_data=higher_tf_data,
    walk_forward_window=2000,
    lookback_window=LOOKBACK_WINDOW_SIZE
)


windows = create_walk_forward_trainer(
    total_steps=80000,
    window_size=2000,
    overlap=200
)


window = windows[0]
env.set_walk_forward_window(window['train_start'], window['train_end'])

current_state = env.reset()


x_low, x_high, pos_info = state_to_tensor(current_state)

Set training window: 0-2000


In [23]:
q_network = QModel(
    input_dim_low=x_low.shape[1], 
    input_dim_high=x_high.shape[1],
    pos_info_dim=extra.shape[0]
)


target_q_network = QModel(
    input_dim_low=x_low.shape[1], 
    input_dim_high=x_high.shape[1],
    pos_info_dim=extra.shape[0]
)

In [24]:
experience = namedtuple("Experience", field_names=["state", "action", "reward", "next_state", "done"])

In [25]:
def compute_loss():
    pass

In [26]:
@torch.function
def agent_learn():
    pass

AttributeError: module 'torch' has no attribute 'function'

In [ ]:
start = time.time()

num_episodes = 80000/2000
max_num_timesteps = 100 # 1000

In [ ]:
80000/2000

40.0